# ER-CyRIS Siklus 2 — Kunci Partisi UNSW dan Holdout Baru

Notebook ini memperbaiki masalah partisi empat arah sebelumnya: seluruh 30.000 sampelnya sudah terpapar oleh evaluasi M0. Tahap ini membuat train, calibration, dan development **hanya dari UNSW-NB15_1.csv**, lalu menyisihkan holdout baru dari `UNSW-NB15_2.csv` setelah membuang grup dengan 43 fitur model identik terhadap **seluruh** berkas pertama dan deduplikasi berkas kedua. Pembagian ini tidak melatih model, memilih ambang, atau menghitung metrik pada holdout.

**Batas inferensi:** berkas kedua berarti pemisahan berdasarkan berkas sumber. Audit rentang waktu dan jumlah IP bersama disimpan sebagai diagnostik. Pemisahan berkas tidak dengan sendirinya membuktikan generalisasi antarorganisasi, independensi host, atau independensi waktu. `M0–M4` pada notebook ablasi lama tidak otomatis sama dengan modul `M1–M4` di arsitektur Bab III. Sumber, hasil lama, dan notebook lama tidak diubah.

Sumber tetap berada di `MyDrive/ercyris_siklus2/`. Output baru ditulis sekali ke `unsw_locked_protocol_v1/`. Bila suatu lock sudah ada, sel akhir berhenti untuk mencegah penimpaan diam diam. Kirim hanya `unsw_lock_summary.json` setelah semua sel selesai.

In [ ]:
import csv, hashlib, json, platform
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/ercyris_siklus2')
except ImportError:
    BASE=Path('.')
P1=BASE/'UNSW-NB15_1.csv'
P2=BASE/'UNSW-NB15_2.csv'
OLD=BASE/'unsw_corrected_v1'
OLD_MANIFEST=OLD/'unsw_corrected_split_manifest.csv'
OLD_JSON=OLD/'unsw_corrected_decision.json'
OUT=BASE/'unsw_locked_protocol_v1'
LOCK=OUT/'unsw_lock_summary.json'
MANIFEST=OUT/'unsw_locked_split_manifest.csv'
SEED=20260926
HOLDOUT_N=4500
CHUNK=100000
for required in (P1,P2,OLD_MANIFEST,OLD_JSON):
    if not required.is_file(): raise FileNotFoundError(f'Dibutuhkan: {required}')
if LOCK.exists() or MANIFEST.exists():
    raise FileExistsError('Partisi terkunci sudah ada. Jangan timpa; gunakan versi lama atau rancangkan versi protokol baru secara eksplisit.')
print('File sumber dan manifest tersedia. Belum ada lock; data mentah tidak diekspor.')

## 1. Validasi sumber dan identitas sampel lama

Contoh baru tidak dipilih dari hasil evaluasi M0. Sampel yang lama hanya digunakan sebagai data pengembangan. Skema 49 kolom mengikuti notebook Siklus 1. SHA-256 sumber berkas pertama harus cocok dengan hasil audit terbaru.

In [ ]:
COLS=['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
      'sttl','dttl','sloss','dloss','service','sload','dload','spkts','dpkts',
      'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth',
      'res_bdy_len','sjit','djit','stime','ltime','sintpkt','dintpkt',
      'tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl',
      'ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst',
      'ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
      'ct_dst_src_ltm','attack_cat','label']
EXCLUDE={'label','attack_cat','srcip','dstip','stime','ltime'}
FEATURES=[c for c in COLS if c not in EXCLUDE]
CATS=['proto','state','service']; NUMS=[c for c in FEATURES if c not in CATS]
assert len(COLS)==49 and len(FEATURES)==43

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def assert_no_header(path):
    with path.open('r',encoding='utf-8-sig',errors='replace',newline='') as stream:
        record=next(csv.reader(stream))
    if len(record)!=49 or record[-1].strip() not in ('0','1'):
        raise ValueError(f'Skema 49 kolom tanpa header tidak cocok: {path.name}')
for path in (P1,P2): assert_no_header(path)
prior=json.loads(OLD_JSON.read_text(encoding='utf-8'))
sha1=sha256(P1)
if sha1!=prior['source_sha256']:
    raise ValueError('Sumber file1 berubah sejak eksperimen M0; berhenti sebelum membuat split.')
old=pd.read_csv(OLD_MANIFEST,dtype={'feature_group_hash64':str})
required={'scheme','partition','source_row_zero_based','label','feature_group_hash64'}
if set(old.columns)!=required or set(old.scheme)!={'80_20','55_15_15_15'}:
    raise ValueError('Manifest eksperimen sebelumnya tidak sesuai; STOP.')
a=old.loc[old.scheme=='80_20',['source_row_zero_based','label']]
b=old.loc[old.scheme=='55_15_15_15',['source_row_zero_based','label']]
if len(a)!=30000 or len(b)!=30000 or a.source_row_zero_based.duplicated().any() or b.source_row_zero_based.duplicated().any():
    raise ValueError('Sampel 30.000 tidak lengkap atau indeks duplikat; STOP.')
sample=a.set_index('source_row_zero_based')['label'].sort_index()
if not sample.equals(b.set_index('source_row_zero_based')['label'].sort_index()):
    raise ValueError('Indeks dan label sampel dua skema berbeda; STOP.')
want_index=set(map(int,sample.index))
print('SHA file1 cocok; 30.000 indeks sampel lama tervalidasi. 4.498 baris holdout lama tidak dipakai sebagai holdout.')

## 2. Streaming dua berkas dan penyaringan kesamaan fitur

Pembuatan grup mengubah 40 kolom numerik menjadi float64 (`NaN` untuk nilai numerik yang gagal dibaca), sementara tiga kolom kategori dipertahankan sebagai teks. Kunci hash memakai 43 fitur yang benar benar masuk model dan format yang sama pada kedua berkas. Benturan hash 64 bit sangat jarang, tetapi pemeriksaan ini tidak dapat mendeteksi kesamaan peristiwa berbeda representasi. Nilai IP dan rentang `stime` hanya dipakai untuk ringkasan tumpang tindih, tidak dimasukkan sebagai fitur model.

In [ ]:
def chunks(path):
    return pd.read_csv(path,header=None,names=COLS,dtype=str,na_filter=False,
                       chunksize=CHUNK,low_memory=False)

def canonical_groups(frame):
    v=frame[FEATURES].copy()
    for c in NUMS:v[c]=pd.to_numeric(v[c],errors='coerce').astype(np.float64)
    for c in CATS:v[c]=v[c].astype(str).replace('', '__MISSING__')
    return pd.util.hash_pandas_object(v,index=False).to_numpy(dtype=np.uint64)

def labels(frame,path):
    lab=pd.to_numeric(frame['label'],errors='coerce')
    if lab.isna().any() or not lab.isin([0,1]).all():
        raise ValueError(f'{path.name}: terdapat label tidak biner; STOP.')
    return lab.to_numpy(dtype=np.int8)

def update_time_host(frame,acc):
    t=pd.to_numeric(frame['stime'],errors='coerce')
    if t.notna().any():
        a,b=float(t.min()),float(t.max())
        acc['time_min']=min(acc['time_min'],a);acc['time_max']=max(acc['time_max'],b)
    for c in ('srcip','dstip'):
        acc['hosts'].update(x for x in frame[c].astype(str) if x)

def new_meta():return {'time_min':float('inf'),'time_max':float('-inf'),'hosts':set()}

source1_groups=set(); sample_groups={};sample_labels={};meta1=new_meta(); n1=0
for frame in chunks(P1):
    yy=labels(frame,P1);gg=canonical_groups(frame)
    idx=np.arange(n1,n1+len(frame),dtype=np.int32)
    for i,g,l in zip(idx,gg,yy):
        if int(i) in want_index:
            sample_groups[int(i)]=int(g);sample_labels[int(i)]=int(l)
    source1_groups.update(map(int,gg))
    update_time_host(frame,meta1);n1+=len(frame)
if len(sample_groups)!=len(want_index) or any(sample_labels[i]!=int(sample.loc[i]) for i in want_index):
    raise ValueError('Indeks/label sampel lama tidak cocok dengan baris file1; STOP.')
if n1!=prior['data']['records']:raise ValueError('Jumlah rekaman file1 berubah; STOP.')
print('Berkas 1:',n1,'baris,',len(source1_groups),'grup fitur; sampel tervalidasi.')

meta2=new_meta();n2=0;seen2=set();cross_rows=0;cross_unique=set()
candidate_idx=[];candidate_group=[];candidate_y=[];duplicate2=0
for frame in chunks(P2):
    yy=labels(frame,P2);gg=canonical_groups(frame)
    for offset,(g,l) in enumerate(zip(gg,yy)):
        g=int(g)
        if g in source1_groups:
            cross_rows+=1;cross_unique.add(g)
        elif g in seen2:
            duplicate2+=1
        else:
            seen2.add(g);candidate_idx.append(n2+offset)
            candidate_group.append(g);candidate_y.append(int(l))
    update_time_host(frame,meta2);n2+=len(frame)
if n2<4500 or len(candidate_idx)<HOLDOUT_N or set(candidate_y)!={0,1}:
    raise ValueError('File2 tidak menyisakan sampel dan dua kelas memadai setelah penyaringan; STOP.')
print('Berkas 2:',n2,'baris; grup bersama file1:',len(cross_unique),
      '| baris lintas-file bersama:',cross_rows,'| kandidat unik tersisa:',len(candidate_idx))

## 3. Bekukan tiga partisi pengembangan dan holdout file kedua

Train, calibration, dan development berasal dari 30.000 observasi file1 yang telah terpapar pada kontrol M0; fungsinya untuk membangun dan memilih aturan baru, **bukan** menguji klaim akhir. Holdout dipilih dari file2 setelah seluruh kesamaan fitur terhadap file1 disingkirkan. Tidak satu pun baris file2 digunakan melatih model di notebook ini. Rasio alami dua kelas dipertahankan saat sampling. Simpan seed dan indeks persis agar seluruh eksperimen berikutnya memakai partisi yang sama.

In [ ]:
idx1=np.array(sorted(want_index),dtype=np.int32)
y1=np.array([sample_labels[int(i)] for i in idx1],dtype=np.int8)
g1=np.array([sample_groups[int(i)] for i in idx1],dtype=np.uint64)
sp=GroupShuffleSplit(n_splits=1,train_size=.60,random_state=SEED)
tr,other=next(sp.split(idx1,y1,g1))
rest=GroupShuffleSplit(n_splits=1,train_size=.50,random_state=SEED+1)
ca_rel,de_rel=next(rest.split(idx1[other],y1[other],g1[other]))
parts={'train':tr,'calibration':other[ca_rel],'development':other[de_rel]}
if sum(map(len,parts.values()))!=len(idx1) or len(set(np.concatenate(list(parts.values()))))!=len(idx1):
    raise AssertionError('Partisi pengembangan tidak lengkap.')
for a in parts:
    pos=parts[a]
    if y1[pos].sum() in (0,len(pos)):raise ValueError(f'Satu kelas di {a}; STOP.')
    for b in parts:
        if a<b and set(map(int,g1[pos]))&set(map(int,g1[parts[b]])):
            raise ValueError(f'Grup fitur identik melintasi {a}/{b}; STOP.')

cand=np.arange(len(candidate_idx),dtype=np.int32)
selected,_=train_test_split(cand,train_size=HOLDOUT_N,random_state=SEED,
                            stratify=np.array(candidate_y,dtype=np.int8))
held_rows=np.array(candidate_idx,dtype=np.int32)[selected]
held_groups=np.array(candidate_group,dtype=np.uint64)[selected]
held_y=np.array(candidate_y,dtype=np.int8)[selected]
if held_y.sum() in (0,len(held_y)) or len(set(map(int,held_groups)))!=HOLDOUT_N:
    raise ValueError('Holdout file2 tidak memenuhi persyaratan kelas/keunikannya; STOP.')
if set(map(int,held_groups)) & source1_groups:
    raise AssertionError('Holdout file2 bocor ke sumber file1; STOP.')
rows=[]
for partition,pos in parts.items():
    for j in pos:
        rows.append({'source_file':P1.name,'partition':partition,'source_row_zero_based':int(idx1[j]),
                     'label':int(y1[j]),'feature_group_hash64':str(int(g1[j]))})
for i,g,yv in zip(held_rows,held_groups,held_y):
    rows.append({'source_file':P2.name,'partition':'holdout',
                 'source_row_zero_based':int(i),'label':int(yv),
                 'feature_group_hash64':str(int(g))})
manifest=pd.DataFrame(rows)
assert len(manifest)==len(idx1)+HOLDOUT_N
statistics={p:{'source_file':P2.name if p=='holdout' else P1.name,
               'n':int(len(z)),'positives':int(z.label.sum()),
               'positive_fraction':float(z.label.mean()),
               'feature_groups':int(z.feature_group_hash64.nunique())}
            for p,z in manifest.groupby('partition')}
print(json.dumps(statistics,indent=2,ensure_ascii=False))

## 4. Simpan berkas beku tanpa melatih atau menguji model

Berkas lock mencatat checksum sumber, unit pemisahan, komposisi partisi, hasil audit silang, dan keterbatasan. Manifest memuat label sebagai kunci audit, tetapi jangan membuka label holdout untuk memilih model, ambang, atau aturan gerbang. Pembacaan pertama holdout untuk metrik dilakukan sekali setelah pipeline dan ambang ditetapkan di development; bila protokol harus diubah, simpan versi baru dan jangan menamai holdout lama belum tersentuh.

In [ ]:
time1=(None if not np.isfinite(meta1['time_min']) else [meta1['time_min'],meta1['time_max']])
time2=(None if not np.isfinite(meta2['time_min']) else [meta2['time_min'],meta2['time_max']])
time_overlap=(None if time1 is None or time2 is None else
              max(time1[0],time2[0])<=min(time1[1],time2[1]))
shared_hosts=len(meta1['hosts']&meta2['hosts'])
sha2=sha256(P2)
OUT.mkdir(parents=True,exist_ok=False)
manifest.to_csv(MANIFEST,index=False)
summary={
 'status':'FROZEN_SPLIT_NO_MODEL_NO_HOLDOUT_EVALUATION',
 'protocol_id':'UNSW_C2_FILE2_V1',
 'sources':{'file1':{'filename':P1.name,'sha256':sha1,'n':int(n1)},
            'file2':{'filename':P2.name,'sha256':sha2,'n':int(n2)}},
 'prior_exposure':'All 30000 file1 development records were already used as train or test in M0 exploratory 80/20.',
 'settings':{'seed':SEED,'holdout_target_n':HOLDOUT_N,'sample_file1_n':int(len(idx1)),
             'feature_count':len(FEATURES),'group_key':'pandas hash of 40 float64 numeric features plus 3 raw string categorical features',
             'exclusions':sorted(EXCLUDE),'split':'group-aware train/calibration/development from file1 30000; no fit on file2',
             'file2_filter':'exclude groups present in all file1 rows; deduplicate remaining file2 feature groups',
             'selection':'stratified sample by binary label among filtered unique file2 groups'},
 'partitions':statistics,
 'file2_filter_audit':{'rows_sharing_model_feature_group_with_file1':int(cross_rows),
                       'unique_feature_groups_shared':int(len(cross_unique)),
                       'rows_duplicate_among_remaining_file2':int(duplicate2),
                       'candidate_unique_groups':int(len(candidate_idx))},
 'context_audit':{'file1_stime_minmax':time1,'file2_stime_minmax':time2,
                  'stime_ranges_overlap':time_overlap,
                  'file1_host_count':int(len(meta1['hosts'])),
                  'file2_host_count':int(len(meta2['hosts'])),
                  'host_identifiers_shared':int(shared_hosts)},
 'manifest':{'filename':MANIFEST.name,'sha256':sha256(MANIFEST),'row_count':int(len(manifest))},
 'runtime':{'python':platform.python_version(),'pandas':pd.__version__,
            'numpy':np.__version__,'sklearn':__import__('sklearn').__version__},
 'not_validated':['host/time independence','M1-M4 framework implementation','H1-H5','gate q_e','risk or response M5-M6'],
 'holdout_rule':'Do not inspect or evaluate file2 holdout for tuning. New protocol required if inspected before freeze.'}
LOCK.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('TERKUNCI:',LOCK)
print('Kirim JSON ringkasan kecil saja. Manifest tetap di Drive untuk rerun; jangan unggah raw CSV.')